# 工程化 RoPE


## 1. the interleaved even/odd version (奇偶交错[**待补充代码实现**])
RoPE 原始论文和 Meta 官方 Llama 2 仓库

## 2. the split-halves version (对半拆分)
Hugging Face transformers 库

In [1]:
import torch

- 定义 `precompute_rope_params` 和 `compute_rope` 方法
$$
f_{\{q,k\}}(\boldsymbol{x}_m, m) = \boldsymbol R_{\Theta,m}^d \boldsymbol W_{\{q,k\}} \boldsymbol x_m \tag{4}
$$
$$
R_{\Theta,m}^d =
\begin{pmatrix}
\cos m\theta_1 & -\sin m\theta_1 & 0 & 0 & \dots & 0 & 0 \\
\sin m\theta_1 & \cos m\theta_1 & 0 & 0 & \dots & 0 & 0 \\
0 & 0 & \cos m\theta_2 & -\sin m\theta_2 & \dots & 0 & 0 \\
0 & 0 & \sin m\theta_2 & \cos m\theta_2 & \dots & 0 & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & 0 & \dots & \cos m\theta_{d/2} & -\sin m\theta_{d/2} \\
0 & 0 & 0 & 0 & \dots & \sin m\theta_{d/2} & \cos m\theta_{d/2}
\end{pmatrix}
$$
$$
\Theta=\{\theta_i = 10000^{-2(i-1)/d},\ i\in[1,2,...,d/2]\}
$$

In [2]:
# 预先计算所有位置、所有维度对应的 cos、sin 旋转系数
def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim)) # `\Theta``

    # Generate position indices
    positions = torch.arange(context_length) # 相当于公式中的 `m`

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # 相当于公式中的 `m\theta`, 但 Shape: (context_length, head_dim // 2) 

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # 相当于公式中的 `m\theta` ， 但 Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

# 拿到 Q/K 张量，用预存的 cos/sin 做旋转变换
def compute_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim), `[:seq_len, :]` 是为了支持短于 context_length 的输入
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

[!NOTE]<br>
1.`torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim` 解释这段代码：
- `torch.arange(0, head_dim, 2)`: `torch.arange(start, end, step)`：从 `start` 开始，**不包含 end**，按步长 `step` 生成数字, 最终形成一维张量
- `[: (head_dim // 2)]`: 属于防御性写法，兼容一些特殊变体 RoPE
---
2. `angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)` 和 `angles = torch.cat([angles, angles], dim=1)`
- `q` 和 `k` 的 shape 为 (seq_len, head_dim)，因为工程化 RoPE 的方法为 `the split-halves version `, 相当于 `q` 和 `k` 会被划分为 shape 为 （seq_len, head_dim // 2）的左右两部分，其中左部分表示实部，右部分表示虚部，左右部分的实部和虚部是两两对应的，对应的实部和虚部构成 `q` 和 `v` 的各个子向量（二维向量）。

2. Applying RoPE to the `q` and `k` tensors

In [3]:
# Settings
batch_size = 2
context_len = 5
num_heads = 4
head_dim = 16

# Instantiate RoPE parameters
cos, sin = precompute_rope_params(head_dim=head_dim, context_length=context_len)

# Dummy query and key tensors
torch.manual_seed(123)
queries = torch.randn(batch_size, num_heads, context_len, head_dim)
keys = torch.randn(batch_size, num_heads, context_len, head_dim)

# Apply rotary position embeddings
queries_rot = compute_rope(queries, cos, sin)
keys_rot = compute_rope(keys, cos, sin)